# ⭐ `fat_not_med` 

**Objetivo**  
Construir a tabela fato **`fat_not_med`** (Notificações × Medicamentos) na camada **Gold** a partir de insumos **Silver**, padronizando posologia (dose/frequência/datas), aplicando regras de normalização de ATC/princípio ativo via `dim_atc` e garantindo integridade referencial para consumo analítico (dashboards, estudos de farmacovigilância, features de modelos).

**Escopo**
- **Entradas (🥈 Silver):**
  - Tabela de notificações/medicamentos com colunas mínimas:  
    `IDENTIFICACAO_NOTIFICACAO`, `PK_MEDICAMENTO`,  
    `CODIGO_ATC`, `PRINCIPIOS_ATIVOS_WHODRUG`, `NOME_MEDICAMENTO_WHODRUG`,  
    `DOSE`, `FREQUENCIA_DOSE`, `INICIO_ADMINISTRACAO`, `FIM_ADMINISTRACAO`,  
    além de atributos clínicos/opcionais (via, forma, lote, etc.).
  - Dimensão **`dim_atc`** (Gold), com: `PK_ATC`, `DESCRICAO`, `CODIGO_ATC`, `PRINCIPIOS_ATIVOS_WHODRUG`.
- **Transformações principais:**
  1) Normalização de chaves textuais (maiúsculas, sem acento).  
  2) Higienização de posologia: `DOSE_NORM`, `FREQUENCIA_DOSE_H`, `INICIO_ADMINISTRACAO_DT`, `FIM_ADMINISTRACAO_DT`.  
  3) Enriquecimento/validação com `dim_atc` (FK: `PK_ATC`).  
  4) Regras de fallback de matching por (`CODIGO_ATC`, `PRINCIPIOS_ATIVOS_WHODRUG`) quando necessário.  
  5) Checks de qualidade, métricas e deduplicação por granularidade.
- **Saída (🥇 Gold):**
  - **`fat_not_med`** (parquet/csv) com:  
    chaves: `IDENTIFICACAO_NOTIFICACAO`, `PK_MEDICAMENTO`, `PK_ATC`  
    medidas/atributos: `DOSE_NORM`, `FREQUENCIA_DOSE_H`, datas normalizadas, `VIA_ADMINISTRACAO`, etc.  
    metadados: `etl_run_id`, `data_processamento`.

**Granularidade & Chaves**
- Granularidade: **Notificação × Medicamento** (uma linha por medicamento notificado dentro da notificação).  
- Chaves: primária lógica composta `IDENTIFICACAO_NOTIFICACAO` + `PK_MEDICAMENTO`.  
- FK para dimensão: `PK_ATC` → `dim_atc.PK_ATC`.

**Usuários & Consumo**
- Times de segurança do paciente/farmacovigilância, BI clínico/fármaco, e *modelagem preditiva* (efeitos, causalidade, severidade, etc.).

**Assunções**
- `dim_atc` já foi construída/validada.  
- Datas no formato `YYYYMMDD` quando vindas de Silver.

**Rastreabilidade**
- Persistir `etl_run_id`, `data_processamento` e métricas de cobertura de `PK_ATC`.


In [1]:
%run ../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_principio_ativo_atc

Exception: File `'../config/bootstrap.py'` not found.

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"

In [4]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)     
bronze.head()

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17
2,BR-ANVISA-300000007,Suspeito,Diamox,Acetazolamide sodium,C03|N03AX|S01EC,None,None,None,Retirada do medicamento,None,None,None,250 milligram (mg),6 hora,250mg a cada 6 horas,None,20181103,20181115,None,oral,None,None,2025-11-17
3,BR-ANVISA-300000007,Suspeito,Hidantal,Phenytoin,N03AB,None,None,None,Retirada do medicamento,None,None,None,None,None,100mg a cada 8 horas,None,20181016,20181115,None,oral,None,None,2025-11-17
4,BR-ANVISA-300000008,Suspeito,Nitroglicerina,Glyceryl trinitrate,C01DA|C05AE,Cristália,None,None,None,None,None,None,None,None,None,None,20181024,None,None,infusão intravenosa,None,None,2025-11-17


In [5]:
bronze.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 23 columns):
 #   Column                                        Non-Null Count   Dtype 
---  ------                                        --------------   ----- 
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object
 1   RELACAO_MEDICAMENTO_EVENTO                    551887 non-null  object
 2   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object
 3   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object
 4   CODIGO_ATC                                    547728 non-null  object
 5   DETENTOR_REGISTRO                             126686 non-null  object
 6   CONCENTRACAO                                  92628 non-null   object
 7   COMPONENTE_SUSPEITO                           56150 non-null   object
 8   ACAO_ADOTADA                                  285540 non-null  object
 9   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-nul

# 🥈 Silver

normalized data, medium quality


## hist_fat_notificacoes_medicamentos

In [6]:
norm = pd.read_csv(Path(project_root) / "data/02_silver/hist_atc/hist_atc.csv")
norm.head()



,PK_MEDICAMENTO,CODIGO_ATC,PRINCIPIOS_ATIVOS_WHODRUG,NOME_MEDICAMENTO_WHODRUG,PK_ATC,DESCRICAO
0,0,A,NaN,Alimentary tract and metabolism,outros,outros
1,1,A01A,Mucin,Saliva,outros,outros
2,2,A01A,Sodium chlorate,Sodium chlorate,outros,outros
3,3,A01AA,Fluoreto de sódio,Dual,outros,outros
4,4,A01AA,Flúor_x000D_|Xilitol,Biotene,outros,outros


In [7]:
# Supondo que no DataFrame `norm` as colunas tenham os mesmos nomes
silver = bronze.merge(
    norm,
    how='left',
    left_on=["CODIGO_ATC", "PRINCIPIOS_ATIVOS_WHODRUG", "NOME_MEDICAMENTO_WHODRUG"],
    right_on=["CODIGO_ATC", "PRINCIPIOS_ATIVOS_WHODRUG", "NOME_MEDICAMENTO_WHODRUG"]
)


silver["INICIO_ADMINISTRACAO"] = pd.to_datetime(silver["INICIO_ADMINISTRACAO"], format='%Y%m%d', errors='coerce')
silver["FIM_ADMINISTRACAO"] = pd.to_datetime(silver["FIM_ADMINISTRACAO"], format='%Y%m%d', errors='coerce')

silver = silver[['IDENTIFICACAO_NOTIFICACAO', 
                                    'RELACAO_MEDICAMENTO_EVENTO',
                                    #'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG',  'CODIGO_ATC',
                                    'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
                                    'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 
                                    'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
                                    'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
                                    'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
                                    'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE',
                                    'PK_MEDICAMENTO', 'PK_ATC']]

silver['ATIVO'] = True 
silver.head()

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,PK_MEDICAMENTO,PK_ATC,ATIVO
0,BR-ANVISA-300000004,Suspeito,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,2018-11-24,NaT,None,infusão intravenosa,None,5833018,9767,outros,True
1,BR-ANVISA-300000005,Suspeito,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,2018-11-22,2018-11-22,None,oral,None,None,16250,outros,True
2,BR-ANVISA-300000007,Suspeito,None,None,None,Retirada do medicamento,None,None,None,250 milligram (mg),6 hora,250mg a cada 6 horas,None,2018-11-03,2018-11-15,None,oral,None,None,5800,outros,True
3,BR-ANVISA-300000007,Suspeito,None,None,None,Retirada do medicamento,None,None,None,None,None,100mg a cada 8 horas,None,2018-10-16,2018-11-15,None,oral,None,None,16515,outros,True
4,BR-ANVISA-300000008,Suspeito,Cristália,None,None,None,None,None,None,None,None,None,None,2018-10-24,NaT,None,infusão intravenosa,None,None,5339,outros,True


## dates

In [8]:
silver['INICIO_ADMINISTRACAO'] = pd.to_datetime(silver['INICIO_ADMINISTRACAO'], errors='coerce')
silver['FIM_ADMINISTRACAO'] = pd.to_datetime(silver['FIM_ADMINISTRACAO'], errors='coerce')

In [9]:
silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 22 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                    551887 non-null  object        
 2   DETENTOR_REGISTRO                             126686 non-null  object        
 3   CONCENTRACAO                                  92628 non-null   object        
 4   COMPONENTE_SUSPEITO                           56150 non-null   object        
 5   ACAO_ADOTADA                                  285540 non-null  object        
 6   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 7   INDICACAO_MEDDRA                              227353 non-null  object        
 8   INDICACAO_RELATADA_NOTIFICADOR_INICIAL        236905 n

In [13]:
silver.to_parquet(Path(project_root) / "data/02_silver/hist_fat_notificacoes_medicamentos/hist_fat_notificacoes_medicamentos.parquet", index=False)

# 🥇 Gold


In [12]:
gold = silver.query("ATIVO==True")
gold.to_parquet(Path(project_root) / "data/03_gold/fat_notificacoes_medicamentos/fat_notificacoes_medicamentos.parquet", index=False)